# Financial-GPT — Global RNN/GRU bidireccional

Modelo global con pesos compartidos y resultados por `cross_key_id`.

**Checkpoint 19:** representación causal, `x_static` train-only y cero dificultad calculada antes del split.


In [ ]:
#@title Configuración general — Financial-GPT
ARCHITECTURE = "rnn_bi"
GLOBAL_MODEL_LABEL = "GLOBAL_RNNBi_E_D"
NOTEBOOK_FILENAME = "code_03_GLOBAL_RNNBi_E_D.ipynb"

# Fuentes
GLOBAL_LONG_URI = ""  # @param {type:"string"}
GLOBAL_LONG_BASE_URI = "s3://ada-us-east-1-sbx-live-mx-m6hn-data/data/sandboxes/m6hn/data/coe_liquidez_diaria/csv/"  # @param {type:"string"}
CALENDAR_URI = "s3://ada-us-east-1-sbx-live-mx-m6hn-data/data/sandboxes/m6hn/data/coe_liquidez_diaria/calendario/calendario.csv"  # @param {type:"string"}
CALENDAR_DATE_COLUMN = "fecha"  # @param {type:"string"}
EXOGENOUS_COLUMNS = []

# Contrato temporal
HORIZON = 25  # @param {type:"integer"}
SEEN_VALIDATION_SIZE = 50  # @param {type:"integer"}
VALIDATION_UNSEEN_FRACTION = 0.15  # @param {type:"number"}
TEST_UNSEEN_FRACTION = 0.15  # @param {type:"number"}
STRIDE = 1  # @param {type:"integer"}
MAX_WINDOW_SIZE = 25  # @param {type:"integer"}

# Presupuesto HPO proxy: muchas configuraciones, poco cómputo por trial
HPO_TRIALS = 150  # @param {type:"integer"}
HPO_EPOCHS = 3  # @param {type:"integer"}
HPO_WINDOWS_PER_SERIES = 4  # @param {type:"integer"}
HPO_VALIDATION_WINDOWS_PER_SERIES = 3  # @param {type:"integer"}
HPO_BATCH = 1024  # @param {type:"integer"}
HPO_REDUCTION_FACTOR = 3  # @param {type:"integer"}
HPO_FINALISTS = 5  # @param {type:"integer"}
HPO_FIDELITY_EPOCHS = 8  # @param {type:"integer"}
HPO_FIDELITY_WINDOWS_PER_SERIES = 8  # @param {type:"integer"}
HPO_TIMEOUT_SECONDS = None

# Entrenamiento productivo
WARM_EPOCHS = 15  # @param {type:"integer"}
WARM_BATCH = 1024  # @param {type:"integer"}
FINE_EPOCHS = 120  # presupuesto total repartido entre niveles curriculares
FINE_BATCH = 1024  # @param {type:"integer"}
CONSOLIDATION_EPOCHS = 3  # @param {type:"integer"}
REPLAY_FRACTION = 0.25  # @param {type:"number"}
FINETUNE_LR_FACTOR = 0.20  # @param {type:"number"}
CONSOLIDATION_LR_FACTOR = 0.10  # @param {type:"number"}
TRAIN_SAMPLES_PER_EPOCH = 32768  # 0 => dataset completo
PATIENCE = 5  # @param {type:"integer"}
NONFINITE_MAX_RETRIES = 3  # @param {type:"integer"}
NONFINITE_LR_FACTOR = 0.20  # @param {type:"number"}
LOSS_FUNCTION = "huber"  # @param ["rmse","mae","mse","smape","wmape","log_cosh","huber"]
SELECTION_METRIC = "robust_macro_mase"  # métrica primaria única para HPO, early stopping y monitor
USE_AUXILIARY_AUTOENCODER = True  # @param {type:"boolean"}
TRAINING_ORDER = "curriculum"  # @param ["curriculum","shuffled"]

# Inferencia, incertidumbre y plots
N_MONTE_CARLO = 100  # @param {type:"integer"}
MC_BATCH_SIZE = 1024  # @param {type:"integer"}
SHOW_PLOTS = True  # @param {type:"boolean"}
PLOT_SERIES = []
PLOT_MAX_SERIES = 50  # 0 => todas
BT_START = ""
BT_END = ""
FC_START = ""
FC_END = ""
FORECAST_BATCH_SIZE = 1024
EXPORT_FORECASTS = True

# Runtime y persistencia
NUM_WORKERS = 0
DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]
SEED = 42
ARTIFACT_ROOT = "./global_runs"
ARTIFACT_S3_URI = "s3://ada-us-east-1-sbx-live-mx-m6hn-data/users/mi31883/financial_gpt"
VERIFY_S3_ROUNDTRIP = True
INSTALL_DEPENDENCIES = False


## 1. Dependencias e imports

En SageMaker, deja `INSTALL_DEPENDENCIES=False` cuando el kernel ya tenga el
entorno del bundle. La instalación opcional no modifica el código fuente.


In [ ]:
if INSTALL_DEPENDENCIES:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "polars>=1.0",
        "pyarrow>=15",
        "torch>=2.1",
        "optuna>=3.5",
        "boto3>=1.34",
    ])


In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import pandas as pd
import polars as pl
import torch
from IPython.display import display

from global_curriculum import GlobalCurriculumConfig
from global_manager import GlobalManager
from global_notebook import (
    GlobalNotebookConfig,
    GlobalNotebookDatasetFactory,
    find_latest_global_long_uri,
    load_global_inputs,
    write_json,
)
from global_training import GlobalHPOConfig, GlobalTrainingConfig

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())


## 2. Resolver fuentes y validar configuración

El split se fija por `account_currency_id`, de modo que saldo y variación de una misma cuenta-divisa nunca caen en particiones distintas. El split ocurre antes de ajustar categorías, estadísticas exógenas o dificultad curricular.


In [ ]:
resolved_global_long_uri = (
    GLOBAL_LONG_URI.strip()
    if GLOBAL_LONG_URI.strip()
    else find_latest_global_long_uri(GLOBAL_LONG_BASE_URI)
)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_name = f"{GLOBAL_MODEL_LABEL}_{run_id}"
run_dir = Path(ARTIFACT_ROOT).expanduser().resolve() / run_name
model_dir = run_dir / "model"
reports_dir = run_dir / "reports"
run_dir.mkdir(parents=True, exist_ok=True)
HPO_STORAGE_URI = f"sqlite:///{(run_dir / 'optuna_study.db').as_posix()}"

notebook_config = GlobalNotebookConfig(
    architecture=ARCHITECTURE,
    global_long_uri=resolved_global_long_uri,
    calendar_uri=CALENDAR_URI,
    artifact_root=str(run_dir),
    horizon=HORIZON,
    seen_validation_size=SEEN_VALIDATION_SIZE,
    validation_unseen_fraction=VALIDATION_UNSEEN_FRACTION,
    test_unseen_fraction=TEST_UNSEEN_FRACTION,
    stride=STRIDE,
    exogenous_columns=tuple(EXOGENOUS_COLUMNS),
    calendar_date_column=CALENDAR_DATE_COLUMN,
    n_trials=HPO_TRIALS,
    hpo_timeout_seconds=HPO_TIMEOUT_SECONDS,
    seed=SEED,
    max_window_size=MAX_WINDOW_SIZE,
    artifact_s3_uri=ARTIFACT_S3_URI,
)
notebook_config.validate()

print("Run:", run_name)
print("Arquitectura:", ARCHITECTURE)
print("global_series_long:", resolved_global_long_uri)
print("calendario:", CALENDAR_URI)
print("artefactos locales:", run_dir)


## 3. Cargar dataset canónico y calendario financiero

`EXOGENOUS_COLUMNS=[]` infiere únicamente columnas numéricas o booleanas del calendario. Estas variables temporales se estandarizan con fechas de train; binarias permanecen 0/1. `tipo_serie` y `divisa` no se repiten en el calendario: se codifican como `x_static` con categorías aprendidas sólo de train, junto con `log_scale` y edad causal.


In [ ]:
inputs = load_global_inputs(notebook_config)

print("global_long shape:", inputs.global_long.shape)
print("series:", inputs.global_long.get_column("cross_key_id").n_unique())
print("rango:", inputs.global_long.get_column("fecha").min(), "→", inputs.global_long.get_column("fecha").max())
print("exogenous_features:", inputs.exogenous_columns)
print("calendar shape:", inputs.calendar.shape)

display(inputs.global_long.head(10))
display(inputs.calendar.head(10))


## 4. Dataset global, holdout temporal y zero-shot split

Para series vistas, las últimas `SEEN_VALIDATION_SIZE` fechas se reservan como
holdout. La ventana de validación puede utilizar historia anterior al corte, pero
sus targets nunca participan en entrenamiento.


In [ ]:
dataset_factory = GlobalNotebookDatasetFactory(
    inputs.global_long,
    inputs.calendar,
    exogenous_columns=inputs.exogenous_columns,
    horizon=HORIZON,
    seen_validation_size=SEEN_VALIDATION_SIZE,
    validation_unseen_fraction=VALIDATION_UNSEEN_FRACTION,
    test_unseen_fraction=TEST_UNSEEN_FRACTION,
    stride=STRIDE,
    seed=SEED,
    max_window_size=MAX_WINDOW_SIZE,
)

print(json.dumps(dataset_factory.summary(), indent=2, ensure_ascii=False))
print("Ejemplo de inicios del holdout seen:")
for series_id, start_date in list(dataset_factory.seen_target_start_dates.items())[:5]:
    print(" ", series_id, "→", start_date)

# Smoke del contrato antes de iniciar HPO.
smoke_bundle = dataset_factory(window_size=3)
print("Smoke windows:")
print("  train:", len(smoke_bundle.train))
print("  validation_seen:", len(smoke_bundle.validation_seen))
print("  validation_unseen:", len(smoke_bundle.validation_unseen))
print("  model_inputs:", tuple(smoke_bundle.train[0]["model_inputs"]))
print("  metadata:", tuple(smoke_bundle.train[0]["metadata"]))
print("Cobertura temporal (primeras series):")
display(dataset_factory.temporal_alignment_report.head(10))


## 5. Configurar HPO proxy, warm-up y fine-tuning curricular

El HPO explora muchas configuraciones con tres epochs, pocas ventanas balanceadas
por serie, un origen de validación por identidad y Hyperband. Los pesos de los
trials se desechan; después se crea un modelo nuevo para warm-up y fine-tuning.


In [ ]:
training_config = GlobalTrainingConfig(
    epochs=HPO_EPOCHS,
    batch_size=HPO_BATCH,
    learning_rate=1e-3,
    weight_decay=1e-5,
    loss=LOSS_FUNCTION,
    patience=PATIENCE,
    min_delta=1e-5,
    grad_clip_norm=1.0,
    scheduler_patience=3,
    scheduler_factor=0.5,
    min_learning_rate=1e-6,
    samples_per_epoch=(
        None if TRAIN_SAMPLES_PER_EPOCH <= 0 else TRAIN_SAMPLES_PER_EPOCH
    ),
    num_workers=NUM_WORKERS,
    seed=SEED,
    device=DEVICE,
    selection_metric=SELECTION_METRIC,
    nonfinite_max_retries=NONFINITE_MAX_RETRIES,
    nonfinite_lr_factor=NONFINITE_LR_FACTOR,
    use_auxiliary_autoencoder=USE_AUXILIARY_AUTOENCODER,
)

hpo_config = GlobalHPOConfig(
    epochs=HPO_EPOCHS,
    windows_per_series_per_epoch=HPO_WINDOWS_PER_SERIES,
    validation_windows_per_series=HPO_VALIDATION_WINDOWS_PER_SERIES,
    objective_metric=SELECTION_METRIC,
    min_resource=1,
    reduction_factor=HPO_REDUCTION_FACTOR,
    finalists=HPO_FINALISTS,
    fidelity_epochs=HPO_FIDELITY_EPOCHS,
    fidelity_windows_per_series_per_epoch=HPO_FIDELITY_WINDOWS_PER_SERIES,
)

curriculum_config = GlobalCurriculumConfig(
    warmup_epochs=WARM_EPOCHS,
    fine_tune_epochs_per_level=max(1, FINE_EPOCHS // max(1, len(set(smoke_bundle.train.series_curriculum_levels.values())) - 1)),
    consolidation_epochs=CONSOLIDATION_EPOCHS,
    replay_fraction=REPLAY_FRACTION,
    fine_tune_lr_factor=FINETUNE_LR_FACTOR,
    consolidation_lr_factor=CONSOLIDATION_LR_FACTOR,
    training_order=TRAINING_ORDER,
)

training_config.validate()
hpo_config.validate()
curriculum_config.validate()

manager = GlobalManager(
    ARCHITECTURE,
    base_training_config=training_config,
    hpo_config=hpo_config,
    curriculum_config=curriculum_config,
    seed=SEED,
)


## 6. Ejecución 5/5 compatible con el pipeline histórico

El motor es global, pero los resultados, outliers y figuras se generan por `cross_key_id`.


In [ ]:
# Compatibilidad de API compacta disponible: manager.fit_global(...)

def print_curriculum_epoch(record):
    print(
        f"[{record.phase:13s}] {record.stage_name:24s} "
        f"epoch={record.global_epoch:03d} "
        f"loss={record.train_loss:.6f} "
        f"objective={record.validation_objective:.6f} "
        f"lr={record.learning_rate:.3e} "
        f"replay={record.replay_samples} "
        f"retries={record.recovery_retries}"
    )

print("=== 1/5 HPO proxy + Warm-up global ===")
manager._warmup_all(
    dataset_factory,
    n_trials=HPO_TRIALS,
    max_epochs=WARM_EPOCHS,
    batch=WARM_BATCH,
    timeout=HPO_TIMEOUT_SECONDS,
    study_name=f"financial_gpt_{ARCHITECTURE}_{run_id}",
    hpo_storage=HPO_STORAGE_URI,
    hpo_load_if_exists=False,
    split_manifest=dataset_factory.split,
    exogenous_columns=inputs.exogenous_columns,
    run_metadata={
        "run_id": run_id,
        "run_name": run_name,
        "global_long_uri": inputs.global_long_uri,
        "calendar_uri": inputs.calendar_uri,
        "notebook": NOTEBOOK_FILENAME,
        "cross_key_is_model_input": False,
        "use_auxiliary_autoencoder": USE_AUXILIARY_AUTOENCODER,
    },
    curriculum_epoch_callback=print_curriculum_epoch,
    show_progress=True,
)

print("\n=== 2/5 Fine-tune curricular ===")
training_result = manager._finetune_all(
    epochs=max(1, FINE_EPOCHS // max(1, len(set(smoke_bundle.train.series_curriculum_levels.values())) - 1)),
    batch=FINE_BATCH,
    curriculum_epoch_callback=print_curriculum_epoch,
    show_progress=True,
)

last_observed = pd.Timestamp(dataset_factory.global_long.get_column("fecha").max())

print("\n=== 3/5 Back-test MC-Dropout ===")
manager._run_backtest(
    n_mc=N_MONTE_CARLO,
    batch_size=MC_BATCH_SIZE,
    show_progress=True,
)

print("\n=== 4/5 Forecast futuro MC-Dropout ===")
if bool(FC_START.strip()) != bool(FC_END.strip()):
    raise ValueError("FC_START y FC_END deben definirse juntos o dejarse ambos vacíos")
if FC_START.strip():
    manager._run_forecast(
        start_date=FC_START,
        end_date=FC_END,
        n_mc=N_MONTE_CARLO,
        batch_size=MC_BATCH_SIZE,
    )
else:
    manager._run_forecast(
        n_steps=HORIZON,
        n_mc=N_MONTE_CARLO,
        batch_size=MC_BATCH_SIZE,
    )

fc_start = pd.Timestamp(manager._df_forecasts["date"].min())
fc_end = pd.Timestamp(manager._df_forecasts["date"].max())

df_bt = manager._backtest_results["df_regression"]
bt_start = BT_START.strip() or pd.Timestamp(df_bt["date"].min()).strftime("%Y-%m-%d")
bt_end = BT_END.strip() or last_observed.strftime("%Y-%m-%d")
plot_series = list(PLOT_SERIES)
if not plot_series:
    plot_series = sorted(manager._future_results)
if PLOT_MAX_SERIES > 0:
    plot_series = plot_series[:PLOT_MAX_SERIES]

print("\n=== 5/5 Visualización ===")
if SHOW_PLOTS:
    manager.visualise(
        bt_start=bt_start,
        bt_end=bt_end,
        fc_start=fc_start.strftime("%Y-%m-%d"),
        fc_end=fc_end.strftime("%Y-%m-%d"),
        series_ids=plot_series,
    )
else:
    print("Visualización desactivada; los dataframes fueron generados.")

results = manager.legacy_results()
print(json.dumps(manager.run_summary().to_dict(), indent=2, ensure_ascii=False))
print("Best candidate:")
print(json.dumps(manager.best_candidate, indent=2, ensure_ascii=False))


## 7. Métricas seen/unseen y resultados por serie

El backtest legacy incluye train/test; estas métricas mantienen además los gates globales seen y zero-shot.


In [ ]:
seen_metrics = manager.backtest_seen(batch_size=FORECAST_BATCH_SIZE)
validation_unseen_metrics = manager.backtest_unseen(batch_size=FORECAST_BATCH_SIZE)

test_unseen_dataset = dataset_factory.build_test_unseen(
    manager.dimensions.window_size
)
test_unseen_metrics = manager.evaluate(
    test_unseen_dataset,
    batch_size=FORECAST_BATCH_SIZE,
)

metrics_table = pl.DataFrame([
    {"partition": "validation_seen", **{k: v for k, v in seen_metrics.to_dict().items() if k != "per_series"}},
    {"partition": "validation_unseen", **{k: v for k, v in validation_unseen_metrics.to_dict().items() if k != "per_series"}},
    {"partition": "test_unseen", **{k: v for k, v in test_unseen_metrics.to_dict().items() if k != "per_series"}},
])
display(metrics_table)


## 8. Persistencia y reportes

El subdirectorio `model/` conserva exactamente los seis artefactos definidos por
`GlobalManager`. Los forecasts y resúmenes se escriben separadamente en
`reports/`.


In [ ]:
run_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
manager.save_model(model_dir)

write_json(reports_dir / "notebook_config.json", notebook_config.to_dict())
write_json(reports_dir / "dataset_summary.json", dataset_factory.summary())
write_json(reports_dir / "scaler_contract.json", dataset_factory.scaler_contract)
write_json(reports_dir / "static_feature_contract.json", dataset_factory.static_feature_contract)
write_json(reports_dir / "exogenous_contract.json", dataset_factory.exogenous_contract)
dataset_factory.difficulty_manifest.write_parquet(
    reports_dir / "difficulty_train_only.parquet"
)
dataset_factory.mase_scale_manifest.write_parquet(
    reports_dir / "mase_scale_causal.parquet"
)
write_json(reports_dir / "best_candidate.json", manager.best_candidate)
dataset_factory.eligibility_manifest.write_parquet(
    reports_dir / "eligibility_manifest.parquet"
)
dataset_factory.temporal_alignment_report.write_parquet(
    reports_dir / "temporal_alignment_report.parquet"
)
write_json(
    reports_dir / "backtest_run_report.json",
    manager._backtest_results["run_report"].to_dict(),
)

execution_config = {
    "n_monte_carlo": N_MONTE_CARLO,
    "hpo_trials": HPO_TRIALS,
    "hpo_epochs": HPO_EPOCHS,
    "hpo_windows_per_series": HPO_WINDOWS_PER_SERIES,
    "hpo_batch": HPO_BATCH,
    "hpo_validation_windows_per_series": HPO_VALIDATION_WINDOWS_PER_SERIES,
    "hpo_finalists": HPO_FINALISTS,
    "hpo_fidelity_epochs": HPO_FIDELITY_EPOCHS,
    "hpo_fidelity_windows_per_series": HPO_FIDELITY_WINDOWS_PER_SERIES,
    "warm_epochs": WARM_EPOCHS,
    "warm_batch": WARM_BATCH,
    "fine_epochs_total": FINE_EPOCHS,
    "fine_batch": FINE_BATCH,
    "loss_function": LOSS_FUNCTION,
    "selection_metric": SELECTION_METRIC,
    "use_auxiliary_autoencoder": USE_AUXILIARY_AUTOENCODER,
    "training_order": TRAINING_ORDER,
    "representation_checkpoint": 19,
    "training_methodology_checkpoint": 20,
    "context_mask_is_model_input": False,
    "nonfinite_max_retries": NONFINITE_MAX_RETRIES,
    "nonfinite_lr_factor": NONFINITE_LR_FACTOR,
    "hpo_storage_uri": HPO_STORAGE_URI,
    "show_plots": SHOW_PLOTS,
}
write_json(reports_dir / "execution_config.json", execution_config)

hpo_trial_rows = []
fidelity_scores = manager.hpo_result.fidelity_scores or {}
selected_trial_number = manager.hpo_result.selected_trial_number
for trial in manager.hpo_result.study.trials:
    hpo_trial_rows.append({
        "number": int(trial.number),
        "state": trial.state.name,
        "proxy_value": None if trial.value is None else float(trial.value),
        "medium_fidelity_score": fidelity_scores.get(int(trial.number)),
        "selected_medium_fidelity": int(trial.number) == selected_trial_number,
        "params_json": json.dumps(trial.params, ensure_ascii=False, sort_keys=True),
        "user_attrs_json": json.dumps(
            trial.user_attrs, ensure_ascii=False, sort_keys=True
        ),
    })
pl.DataFrame(hpo_trial_rows).write_parquet(reports_dir / "hpo_trials.parquet")
write_json(
    reports_dir / "evaluation_metrics.json",
    {
        "validation_seen": seen_metrics.to_dict(),
        "validation_unseen": validation_unseen_metrics.to_dict(),
        "test_unseen": test_unseen_metrics.to_dict(),
    },
)
metrics_table.write_parquet(reports_dir / "evaluation_metrics.parquet")

if EXPORT_FORECASTS:
    seen_forecast = manager.forecast(
        manager.datasets.validation_seen,
        batch_size=FORECAST_BATCH_SIZE,
    )
    validation_unseen_forecast = manager.forecast(
        manager.datasets.validation_unseen,
        batch_size=FORECAST_BATCH_SIZE,
    )
    test_unseen_forecast = manager.forecast(
        test_unseen_dataset,
        batch_size=FORECAST_BATCH_SIZE,
    )
    seen_forecast.write_parquet(reports_dir / "forecast_validation_seen.parquet")
    validation_unseen_forecast.write_parquet(
        reports_dir / "forecast_validation_unseen.parquet"
    )
    test_unseen_forecast.write_parquet(
        reports_dir / "forecast_test_unseen.parquet"
    )


# Contrato legacy por serie.
manager._backtest_results["df_regression"].to_parquet(
    reports_dir / "backtest_mc_by_series.parquet", index=False
)
manager._backtest_results["df_regression_metrics"].to_parquet(
    reports_dir / "backtest_metrics_by_series.parquet", index=False
)
manager._df_forecasts.to_parquet(
    reports_dir / "future_forecast_mc_by_series.parquet", index=False
)
manager._df_outliers.reset_index().to_parquet(
    reports_dir / "hierarchical_outliers_by_series.parquet", index=False
)

uploaded_uri = None
loaded_from_s3 = None
if ARTIFACT_S3_URI.strip():
    uploaded_uri = manager.save_model_s3(
        ARTIFACT_S3_URI,
        run_id=run_name,
        reports_dir=reports_dir,
        update_latest=True,
    )
    if VERIFY_S3_ROUNDTRIP:
        loaded_from_s3 = GlobalManager.load_model_s3(
            uploaded_uri,
            map_location="cpu",
        )
        if loaded_from_s3.run_summary().state_digest != manager.run_summary().state_digest:
            raise RuntimeError("S3 roundtrip state digest mismatch")

print("Run local:", run_dir)
print("Modelo:", model_dir)
print("Reportes:", reports_dir)
if uploaded_uri:
    print("Run S3 comprometido:", uploaded_uri)
    print("Latest:", manager.last_s3_save_result.latest_uri)
    print("Load S3 verificado:", bool(loaded_from_s3 is not None))


## 9. Gate final del run

Un run válido conserva una sola arquitectura, un solo `state_dict`, el orden exacto de variables temporales y estáticas, el scaler causal lineal y métricas independientes para seen y unseen. Los modelos del checkpoint 18 no son compatibles dimensionalmente y deben reentrenarse.


In [ ]:
required_model_files = {
    "manifest.json",
    "model_state.pt",
    "metrics.json",
    "history.json",
    "hpo_summary.json",
    "split_manifest.json",
}
actual_model_files = {path.name for path in model_dir.iterdir() if path.is_file()}
assert actual_model_files == required_model_files
assert manager.is_fitted
assert not isinstance(manager.model, dict)
assert manager.dimensions.exogenous_columns == tuple(inputs.exogenous_columns)
assert manager.run_metadata["cross_key_is_model_input"] is False
assert seen_metrics.num_series > 0
assert validation_unseen_metrics.num_series > 0
assert test_unseen_metrics.num_series > 0
assert type(manager.hpo_result.study.pruner).__name__ == "HyperbandPruner"
assert manager.hpo_config.objective_metric == SELECTION_METRIC
assert manager.training_result.training_config.selection_metric == SELECTION_METRIC
assert manager.training_result.training_config.loss == LOSS_FUNCTION
assert (reports_dir / "execution_config.json").is_file()
assert (reports_dir / "hpo_trials.parquet").is_file()
assert (reports_dir / "scaler_contract.json").is_file()
assert (reports_dir / "eligibility_manifest.parquet").is_file()
assert (reports_dir / "mase_scale_causal.parquet").is_file()
assert (reports_dir / "temporal_alignment_report.parquet").is_file()
assert (reports_dir / "backtest_run_report.json").is_file()

print("✅ Un modelo global compartido")
print("✅ cross_key_id fuera del forward")
print("✅ validación temporal seen")
print("✅ validación zero-shot unseen")
print("✅ HPO proxy + selección medium-fidelity de finalistas")
print("✅ objetivo único robust_macro_mase")
print("✅ sampler balanceado por tipo, nivel, serie y ventana")
print("✅ escalamiento lineal causal calculado sólo con y_context")
print("✅ persistencia reproducible")
print("✅ run:", run_name)

assert set(results) == {"warm", "fine", "backtest", "forecast", "df_forecasts", "df_outliers"}
assert not results["backtest"]["df_regression"].empty
assert not results["df_forecasts"].empty
assert set(manager._future_results).issubset(set(inputs.global_long["cross_key_id"].unique()))
print("✅ warm-up y fine-tuning continuos")
print("✅ backtest train/test MC-Dropout por serie")
print("✅ forecast por pasos válidos del TemporalAxis")
print("✅ outliers y tres visualizaciones legacy")
